<a href="https://colab.research.google.com/github/heisdenverr/llm-interpretability/blob/master-branch/SAE_of_Qwen2_0.5B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
torch.manual_seed(42)
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2-0.5B', dtype=torch.float32)
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2-0.5B', dtype=torch.float32)

messages = ["The Eiffel Tower is located in"
]

inputs = tokenizer(messages, return_tensors='pt')

outputs = model(**inputs)
print(outputs.logits.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

torch.Size([1, 8, 151936])


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


torch.Size([1, 26, 151936])-->(B, N_TOKENS, VOCAB_SIZE)


In [4]:
last_token = outputs.logits[0, -1,:]
token_id = torch.argmax(last_token)
decode = tokenizer.decode(token_id)
print(f"Token id: {token_id}")
print(f"Decoded to text: {decode}")

Token id: 12095
Decoded to text:  Paris


In [5]:
tokenizer.decode(torch.argmax(outputs.logits, dim=-1))

[' following-el Tower is the in Paris']

In [6]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [7]:
model.model.layers[12].mlp

Qwen2MLP(
  (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
  (up_proj): Linear(in_features=896, out_features=4864, bias=False)
  (down_proj): Linear(in_features=4864, out_features=896, bias=False)
  (act_fn): SiLUActivation()
)

In [8]:
captured = {}
layers = [9, 10, 11, 12, 13, 14]

def get_hook(layer_id):
  def hook(module, input, output):
    captured[layer_id] = output.cpu().detach()
  return hook

handles = [model.model.layers[n].mlp.register_forward_hook(get_hook(n)) for n in layers]


In [9]:
from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:500]")

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [10]:
text = [text for text in ds['text'] if text.strip()]
len(text)

338

In [11]:
store= {9: [],
        10: [],
        11: [],
        12: [],
        13: [],
        14 :[]
           }

for i in text:
  tokenids = tokenizer(i,
                      return_tensors='pt',
                      truncation=True,
                      max_length=128)
  with torch.no_grad():
    y_pred = model(**tokenids)

  for layer in store:
    store[layer].append(captured[layer])

for handle in handles:
  handle.remove()

In [12]:
len(store[12])

338

In [13]:
store[12][0].shape

torch.Size([1, 8, 896])

In [14]:
activation_9 = torch.cat([t.squeeze(0) for t in store[9]], dim=0)
activation_10 = torch.cat([t.squeeze(0) for t in store[10]], dim=0)
activation_11 = torch.cat([t.squeeze(0) for t in store[11]], dim=0)
activation_12 = torch.cat([t.squeeze(0) for t in store[12]], dim=0)
activation_13 = torch.cat([t.squeeze(0) for t in store[13]], dim=0)
activation_14 = torch.cat([t.squeeze(0) for t in store[14]], dim=0)

In [15]:
activation_10.shape

torch.Size([18639, 896])

In [16]:
from torch import nn

class SAE(nn.Module):

  def __init__(self, in_dim, out_dim):
    super().__init__()
    self.encoder = nn.Linear(in_features=in_dim, out_features=out_dim)
    self.decoder = nn.Linear(in_features=out_dim, out_features=in_dim)
    self.relu = nn.ReLU()

  def forward(self, x: torch.Tensor):

    hidden = self.relu(self.encoder(x))
    reconstruction = self.decoder(hidden)
    return reconstruction, hidden

In [17]:
sparsity = SAE(in_dim=896, out_dim=3584)
sparsity

SAE(
  (encoder): Linear(in_features=896, out_features=3584, bias=True)
  (decoder): Linear(in_features=3584, out_features=896, bias=True)
  (relu): ReLU()
)

In [18]:
sparsity(activation_14)

(tensor([[ 0.0660,  0.0098, -0.0375,  ...,  0.0160, -0.0016,  0.0094],
         [ 0.0293, -0.0201, -0.0650,  ...,  0.0374,  0.0265,  0.0016],
         [-0.0116, -0.0309, -0.0358,  ...,  0.0357,  0.0018,  0.0081],
         ...,
         [ 0.0286,  0.0035, -0.0504,  ...,  0.0004,  0.0503, -0.0527],
         [ 0.0391, -0.0018, -0.0497,  ...,  0.0228,  0.0200,  0.0104],
         [-0.0048,  0.0226, -0.0396,  ...,  0.0734,  0.0272,  0.0107]],
        grad_fn=<AddmmBackward0>),
 tensor([[0.0000, 0.0000, 0.0253,  ..., 0.1533, 0.0000, 0.1134],
         [0.0473, 0.0170, 0.0653,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.1462,  ..., 0.0000, 0.1031, 0.0000],
         ...,
         [0.0443, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0821, 0.0884, 0.0000,  ..., 0.0000, 0.0000, 0.0337],
         [0.0000, 0.0920, 0.0587,  ..., 0.0000, 0.0000, 0.0000]],
        grad_fn=<ReluBackward0>))

In [19]:
from torch.utils.data import DataLoader

In [20]:
train_dataloader = DataLoader(activation_14, batch_size=256)
tok = next(iter(train_dataloader))
tok.shape

torch.Size([256, 896])

In [21]:
tok.shape

torch.Size([256, 896])

In [22]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(sparsity.parameters(), lr=1e-4)
lambda_val  = 5e-1
for epoch in range(55):
  total_loss = 0
  for x in train_dataloader:
    optimizer.zero_grad()
    reconstrunction, hidden = sparsity(x)
    loss = loss_fn(reconstrunction, x) + lambda_val * hidden.abs().mean()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch {epoch+1} loss: {total_loss / len(train_dataloader):.4f}")

Epoch 1 loss: 0.0489
Epoch 2 loss: 0.0385
Epoch 3 loss: 0.0350
Epoch 4 loss: 0.0322
Epoch 5 loss: 0.0300
Epoch 6 loss: 0.0281
Epoch 7 loss: 0.0264
Epoch 8 loss: 0.0250
Epoch 9 loss: 0.0236
Epoch 10 loss: 0.0225
Epoch 11 loss: 0.0215
Epoch 12 loss: 0.0206
Epoch 13 loss: 0.0197
Epoch 14 loss: 0.0190
Epoch 15 loss: 0.0183
Epoch 16 loss: 0.0177
Epoch 17 loss: 0.0172
Epoch 18 loss: 0.0167
Epoch 19 loss: 0.0162
Epoch 20 loss: 0.0158
Epoch 21 loss: 0.0154
Epoch 22 loss: 0.0151
Epoch 23 loss: 0.0148
Epoch 24 loss: 0.0144
Epoch 25 loss: 0.0142
Epoch 26 loss: 0.0139
Epoch 27 loss: 0.0136
Epoch 28 loss: 0.0134
Epoch 29 loss: 0.0131
Epoch 30 loss: 0.0129
Epoch 31 loss: 0.0127
Epoch 32 loss: 0.0125
Epoch 33 loss: 0.0123
Epoch 34 loss: 0.0121
Epoch 35 loss: 0.0119
Epoch 36 loss: 0.0117
Epoch 37 loss: 0.0116
Epoch 38 loss: 0.0114
Epoch 39 loss: 0.0113
Epoch 40 loss: 0.0111
Epoch 41 loss: 0.0110
Epoch 42 loss: 0.0108
Epoch 43 loss: 0.0107
Epoch 44 loss: 0.0106
Epoch 45 loss: 0.0105
Epoch 46 loss: 0.01

In [23]:

with torch.no_grad():
  batch = next(iter(train_dataloader))
  _, hidden = sparsity(batch)
  sparsity_ratio = (hidden == 0).float().mean()
  print(f"Sparsity: {sparsity_ratio:.4f}")

Sparsity: 0.8665


In [24]:
with torch.no_grad():
  _, hidden = sparsity(activation_14)
n_0 = hidden[:, 0]
top10 = torch.topk(n_0, 10)
print(top10.indices)

tensor([ 9858,  9916,  9932,  9842, 11861,  9742,  9821,  9872,  9498,  9802])


In [25]:
sum(t.shape[1] for t in store[14])

18639

In [26]:
lengths = torch.tensor([t.shape[1] for t in store[14]])

In [27]:
lengths

tensor([  8, 128, 110, 116,   6, 128, 128,  91,   6, 128, 128, 128,   6, 128,
        128,   8, 128,   8, 128, 107,   6,  86, 128, 128, 103,   6,  80,   9,
        128, 128, 128,  10, 128, 102,   6, 128,   7, 128,  96,  18, 128, 128,
         48,  17,  23,  23,  86,  93,  88,  23,  20,  20,  21,  12,  16,  11,
         10,  18,  11, 128,  99,  72, 128, 128, 128, 128, 106,  96,  64,  26,
         16,  17,   9,  17,  16,   7,   4,  21,  13,   6,  24,   6,   6,   4,
          9, 101, 128,  46,  28,   8, 128, 128,  10, 126,  68,   7, 128,  56,
        128, 128,   7, 109,  94,  99,   6,   9,  99,  84,  13,  65,  99,  92,
         19, 109, 110,  10, 128, 128,   9,  45,  95, 128,  11, 108,  86,   6,
         87,  91,  83,  11, 128,  13, 128, 128,   6,   8,  17,  17,  20,  20,
         20,  15,  17,  22,  27,  13,   8,  16,  14,  16,  25,  16,  14,  17,
         14,  16,  27,  24,  17,  15,  14,  17,  22,  19,  17,  16,  17,  23,
         16,  22,  17,  18,  14,  15,  14,  15,  17,  13,  16,  

In [28]:
boundaries = torch.cumsum(lengths, dim=0)
print(boundaries[:10])

tensor([  8, 136, 246, 362, 368, 496, 624, 715, 721, 849])


In [29]:
top10

torch.return_types.topk(
values=tensor([0.9890, 0.9625, 0.9448, 0.9175, 0.9153, 0.8634, 0.8336, 0.8246, 0.8088,
        0.7574]),
indices=tensor([ 9858,  9916,  9932,  9842, 11861,  9742,  9821,  9872,  9498,  9802]))

In [30]:
index = []
for i in top10.indices:
  index.append((boundaries>i).nonzero()[0])
index

[tensor([182]),
 tensor([185]),
 tensor([186]),
 tensor([181]),
 tensor([222]),
 tensor([175]),
 tensor([180]),
 tensor([183]),
 tensor([161]),
 tensor([179])]

In [32]:
sd_idx = []

_, top10_ = hidden.topk(10, dim=-1)

top10_act = [w.tolist() for w in top10_]


for i in top10_:

  indices = [(boundaries>x).nonzero()[0].item() for x in i]

  sd_idx.append(tuple(n for n in indices))

In [33]:
len(top10_act)

18639

In [34]:

top10_[-1, :]

tensor([1892,  262, 3142, 1769, 2359,  321, 2555, 2293,  372, 1934])

In [35]:
xxd = [w.tolist() for w in top10_]

In [36]:
import pandas as pd

df = pd.DataFrame({
    'sentence index': sd_idx,
    'feature indices': top10_act
})
df.head()

,sentence index,feature indices
0,"(30, 13, 41, 1, 41, 28, 14, 21, 41, 41)","[2659, 1158, 3486, 45, 3549, 2414, 1289, 1778,..."
1,"(37, 5, 26, 33, 14, 31, 19, 1, 30, 41)","[3086, 390, 2249, 2864, 1304, 2682, 1743, 62, ..."
2,"(3, 10, 24, 41, 7, 30, 21, 21, 21, 13)","[305, 960, 2112, 3485, 707, 2601, 1752, 1777, ..."
3,"(35, 1, 35, 7, 7, 10, 14, 21, 13, 29)","[2941, 12, 2939, 707, 650, 974, 1304, 1837, 11..."
4,"(1, 5, 26, 22, 16, 1, 1, 25, 38, 6)","[77, 376, 2220, 1839, 1492, 57, 12, 2202, 3255..."


In [183]:
feature_one_interpretability = pd.DataFrame({
    'sentence index per neuron':[i for i in df['sentence index'][0]],
    'feature indices per neuron': [i for i in df['feature indices'][0]],
    'sentence': [text[i] for i in df['sentence index'][0]]
    })

In [184]:
feature_one_interpretability.head()

,sentence index per neuron,feature indices per neuron,sentence
0,30,2659,"Two manga adaptations were produced , followi..."
1,13,1158,Concept work for Valkyria Chronicles III bega...
2,41,3486,This movement is prompted by the feeling that...
3,1,45,Senjō no Valkyria 3 : Unrecorded Chronicles (...
4,41,3549,This movement is prompted by the feeling that...


In [185]:
hidden[:, 2659].topk(10)

torch.return_types.topk(
values=tensor([0.9732, 0.9729, 0.9716, 0.9716, 0.9712, 0.9712, 0.9712, 0.9707, 0.9704,
        0.9700]),
indices=tensor([3845, 8900, 9190, 9168, 9059, 9076, 9096, 9217, 4223, 9356]))

In [186]:
_s = []
_dec = []
for k in feature_one_interpretability['feature indices per neuron']:
  t_v, t_k = hidden[:, k].topk(10)
  _s.append(tuple(i.item() for i in t_k))
  _dec.append(tokenizer.decode(all_token_ids[t_k]))
feature_one_interpretability['feature indices per neuron decoded'] = [tokenizer.decode(all_token_ids[i]) for i in feature_one_interpretability['feature indices per neuron']]
feature_one_interpretability['all_activ_neuron'] = _s

In [187]:
_dec

[' Inside Religious Christmas Christmas National Shakespeare Shakespeare Beautiful Most Autumn',
 ' Shakespeare Shakespeare Inside Autumn National Kurt Columbus Columbus Summer Time',
 ' but due partly Inside Shakespeare Shakespeare Kurt Autumn Christmas Christmas',
 ' Christmas Christmas Religious Autumn National Shakespeare Shakespeare Kurt Summer Beautiful',
 ' , , , , , , , , , ,',
 ' . . . . . . . . . .',
 ' 1  1  cases Shakespeare Shakespeare Inside',
 ' Return Shakespeare Shakespeare PlayStation With Christmas Christmas Inside Time After',
 ' , first Inside Shakespeare Shakespeare Religious Christmas Christmas Autumn Columbus',
 ' extent and while  to be  to from birth']

In [192]:
feature_one_interpretability.head()

,sentence index per neuron,feature indices per neuron,sentence,feature indices per neuron decoded,all_activ_neuron
0,30,2659,"Two manga adaptations were produced , followi...",1,"(3845, 8900, 9190, 9168, 9059, 9076, 9096, 921..."
1,13,1158,Concept work for Valkyria Chronicles III bega...,",","(9076, 9096, 3845, 9356, 9059, 2203, 13681, 12..."
2,41,3486,This movement is prompted by the feeling that...,aspect,"(8573, 979, 985, 3845, 9096, 9076, 2203, 9356,..."
3,1,45,Senjō no Valkyria 3 : Unrecorded Chronicles (...,),"(9168, 9190, 8900, 9356, 9059, 9096, 9076, 220..."
4,41,3549,This movement is prompted by the feeling that...,charge,"(4805, 9016, 6774, 10134, 8011, 6777, 10095, 1..."


In [190]:
import os

path = 'model/result/interpretations'

os.makedirs(path, exist_ok=True)

feature_one_interpretability.to_pickle(os.path.join(path, 'feature_one_interpretability.pkl'))
print(f"save successfully")

save successfully


In [198]:
# So i passed the dataset to a llm, to get a one line label
label = {
"Capitalized poetic title words in Japanese media adaptations",
"Comma punctuation in dense multi-clause game development prose",
"Formal abstract nouns in 19th century legal authority language",
"Closing parentheses in bilingual Japanese-English title translations",
"Words of authority and custody in formal governmental demand language",
"Time period references in entertainment media release scheduling language",
"Military mission terminology in video game design and development context",
"Sales and commercial performance metrics in Japanese gaming charts",
"Definite article 'the' in formal governmental possession and authority claims",
"Collective civic nouns in 19th century state rights political discourse"
}
"""
Neuron 2659 (token: "1", sentence: Valkyria manga adaptations)
"Capitalized poetic title words in Japanese media adaptations"
Neuron 1158 (token: ",", sentence: Valkyria Chronicles III development)
"Comma punctuation in dense multi-clause game development prose"
Neuron 3486 (token: "aspect", sentence: Little Rock Arsenal speech)
"Formal abstract nouns in 19th century legal authority language"
Neuron 45 (token: ")", sentence: Valkyria Chronicles III intro)
"Closing parentheses in bilingual Japanese-English title translations"
Neuron 3549 (token: "charge", sentence: Little Rock Arsenal speech)
"Words of authority and custody in formal governmental demand language"
Neuron 2414 (token: "period", sentence: Valkyria Chronicles anime)
"Time period references in entertainment media release scheduling language"
Neuron 1289 (token: "mission", sentence: Valkyria Chronicles III development)
"Military mission terminology in video game design and development context"
Neuron 1778 (token: "sales", sentence: Valkyria Chronicles III sales)
"Sales and commercial performance metrics in Japanese gaming charts"
Neuron 3551 (token: "the", sentence: Little Rock Arsenal speech)
"Definite article 'the' in formal governmental possession and authority claims"
Neuron 3438 (token: "citizens", sentence: Little Rock Arsenal speech)
"Collective civic nouns in 19th century state rights political discourse"

Pattern across all 10: Your SAE is dominated by two topics — Valkyria Chronicles and Little Rock Arsenal — confirming the corpus bias finding. But within those topics you're seeing genuinely specific features: punctuation style, syntactic register, domain vocabulary, and even function words in specific contexts.

"""

'\nNeuron 2659 (token: "1", sentence: Valkyria manga adaptations)\n"Capitalized poetic title words in Japanese media adaptations"\nNeuron 1158 (token: ",", sentence: Valkyria Chronicles III development)\n"Comma punctuation in dense multi-clause game development prose"\nNeuron 3486 (token: "aspect", sentence: Little Rock Arsenal speech)\n"Formal abstract nouns in 19th century legal authority language"\nNeuron 45 (token: ")", sentence: Valkyria Chronicles III intro)\n"Closing parentheses in bilingual Japanese-English title translations"\nNeuron 3549 (token: "charge", sentence: Little Rock Arsenal speech)\n"Words of authority and custody in formal governmental demand language"\nNeuron 2414 (token: "period", sentence: Valkyria Chronicles anime)\n"Time period references in entertainment media release scheduling language"\nNeuron 1289 (token: "mission", sentence: Valkyria Chronicles III development)\n"Military mission terminology in video game design and development context"\nNeuron 1778 (to

In [199]:
label

{'Capitalized poetic title words in Japanese media adaptations',
 'Closing parentheses in bilingual Japanese-English title translations',
 'Collective civic nouns in 19th century state rights political discourse',
 'Comma punctuation in dense multi-clause game development prose',
 "Definite article 'the' in formal governmental possession and authority claims",
 'Formal abstract nouns in 19th century legal authority language',
 'Military mission terminology in video game design and development context',
 'Sales and commercial performance metrics in Japanese gaming charts',
 'Time period references in entertainment media release scheduling language',
 'Words of authority and custody in formal governmental demand language'}

In [204]:
_featlabel = pd.DataFrame(label)
_featlabel

,0
0,Comma punctuation in dense multi-clause game d...
1,Formal abstract nouns in 19th century legal au...
2,Time period references in entertainment media ...
3,Capitalized poetic title words in Japanese med...
4,Collective civic nouns in 19th century state r...
5,Closing parentheses in bilingual Japanese-Engl...
6,Sales and commercial performance metrics in Ja...
7,Words of authority and custody in formal gover...
8,Definite article 'the' in formal governmental ...
9,Military mission terminology in video game des...


In [206]:
feature_one_interpretability['label'] = _featlabel
feature_one_interpretability.head()

,sentence index per neuron,feature indices per neuron,sentence,feature indices per neuron decoded,all_activ_neuron,label
0,30,2659,"Two manga adaptations were produced , followi...",1,"(3845, 8900, 9190, 9168, 9059, 9076, 9096, 921...",Comma punctuation in dense multi-clause game d...
1,13,1158,Concept work for Valkyria Chronicles III bega...,",","(9076, 9096, 3845, 9356, 9059, 2203, 13681, 12...",Formal abstract nouns in 19th century legal au...
2,41,3486,This movement is prompted by the feeling that...,aspect,"(8573, 979, 985, 3845, 9096, 9076, 2203, 9356,...",Time period references in entertainment media ...
3,1,45,Senjō no Valkyria 3 : Unrecorded Chronicles (...,),"(9168, 9190, 8900, 9356, 9059, 9096, 9076, 220...",Capitalized poetic title words in Japanese med...
4,41,3549,This movement is prompted by the feeling that...,charge,"(4805, 9016, 6774, 10134, 8011, 6777, 10095, 1...",Collective civic nouns in 19th century state r...


In [207]:
_path = 'model/result/interpretations'

feature_one_interpretability.to_pickle(os.path.join(_path, 'full_feature_one_interpretability.pkl'))

In [191]:
from google.colab import files
import os


file_to_download = os.path.join('model/result/interpretations', 'feature_one_interpretability.pkl')


files.download(file_to_download)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [210]:
files.download(os.path.join('model/result/interpretations', 'full_feature_one_interpretability.pkl'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [213]:
!zip -r model_results.zip /content/model

  adding: content/model/ (stored 0%)
  adding: content/model/result/ (stored 0%)
  adding: content/model/result/interpretations/ (stored 0%)
  adding: content/model/result/interpretations/full_feature_one_interpretability.pkl (deflated 53%)
  adding: content/model/result/interpretations/feature_one_interpretability.pkl (deflated 52%)
  adding: content/model/sae_layer14.pt (deflated 7%)


In [211]:
torch.save(sparsity.state_dict(), os.path.join('model', 'sae_layer14.pt'))

In [215]:
files.download('model_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
all_token_ids = torch.cat([
    tokenizer(nb, return_tensors='pt', truncation=True, max_length=128)['input_ids'][0]
    for nb in text
])

In [ ]:
"""

    This takes a sample row from hidden ( 18639, 3584 ), (, 3584)
    Gets top activating neuron
    return top activating neuron for the given token

    so we get a 2d vector (18639, 10), each row is a feature/token  and each row with 10 columns containing top activating neurons in the order of magnitude

    td gives us the token indices for the activating neurons, we get what token get fired/activated, when the model sees a feature token


"""

# tv, td = hidden.topk(10, dim=-1)
# td.shape, tv.shape

In [ ]:
"""
  Input sentence
      ↓ tokenizer
  (1, seq_len)           # token IDs, 1 sentence, seq_len tokens

      ↓ model forward pass
  (1, seq_len, 896)      # hook captures this — batch × tokens × hidden dim

      ↓ squeeze + concat across 338 sentences
  (18639, 896)           # all tokens stacked, each row is one token

      ↓ DataLoader with batch_size=256
  (256, 896)             # 256 tokens at a time fed to SAE

      ↓ SAE encoder + ReLU
  (256, 3584)            # each token now has 3584 sparse feature activations

      ↓ SAE decoder
  (256, 896)             # reconstructed back to original hidden dim

  activations_9          # shape (18639, 896)
  activations_9[0]       # shape (896,)  — first token's full vector
  activations_9[:, 0]    # shape (18639,) — dimension 0 across all tokens

  hidden                 # shape (18639, 3584)
  hidden[:, 0]           # shape (18639,) — neuron 0's activation across all tokens ← this is what you used for interpretation
  hidden[0]              # shape (3584,)  — all neuron activations for first token
"""